<a href="https://colab.research.google.com/github/jensentke/DS-Work-Folder/blob/main/DSSP_Finetuning_2026_Part_1_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. Install compatible Unsloth environment for Colab
# ============================================================


%pip install --upgrade \
    "jedi>=0.16,<0.20" \
    "protobuf>=6.33.5,<7" \
    "fsspec[http]==2025.9.0" \
    "gcsfs==2025.9.0"




In [ ]:
%pip install peft accelerate bitsandbytes sentencepiece huggingface_hub


In [ ]:
!pip install -q -U unsloth

In [ ]:
import torch
import importlib.metadata as md

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for pkg in ["unsloth", "unsloth_zoo", "transformers", "trl", "torch", "datasets", "peft", "bitsandbytes", "protobuf"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except Exception as e:
        print(f"{pkg}: not found")

CUDA available: True
GPU: NVIDIA L4
unsloth: 2026.11.4
unsloth_zoo: 2026.11.4
transformers: 5.17.0
trl: 1.15.0
torch: 2.14.1+cu130
datasets: 4.8.5
peft: 0.21.2
bitsandbytes: 0.50.2
protobuf: 6.33.6


In [ ]:
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

max_seq_length = 2048
dtype = None
load_in_4bit = True

model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 23:00:41.033000 52422 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 23:00:41.113000 52422 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded successfully.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA adapters added.")

Unsloth 2026.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
LoRA adapters added.


In [ ]:
# ============================================================
# Create tiny train/test data
# ============================================================

train_examples = [
    {
        "question": "What is the HW percentage in DS 6371?",
        "answer": "The homework percentage in DS 6371 is 10%."
    },
    {
        "question": "What is the midterm percentage in DS 6371?",
        "answer": "The midterm is worth 25% in DS 6371."
    },
    {
        "question": "What is the final exam worth in DS 6371?",
        "answer": "The final exam is worth 25% of the grade in DS 6371."
    },
    {
        "question": "What does Whamo mean?",
        "answer": "Whamo means Excitement, Respect, and Celebration of Hard Work."
    },
]

# Repeat examples to give the tiny demo a little more signal
train_examples = train_examples * 3

test_examples = [
    {
        "question": "What percentage of the grade is the homework worth in DS6371?",
        "answer": "The homework is worth 10% of the grade in DS 6371."
    },
    {
        "question": "What is 5 + 7?",
        "answer": "5 + 7 equals 12."
    },
    {
        "question": "What is the midterm percentage in DS 6371?",
        "answer": "The midterm is worth 25% in DS 6371."
    },
    {
        "question": "What is the weight of the final exam in DS 6371?",
        "answer": "The final exam is worth 25% of the grade in DS 6371."
    },
    {
        "question": "What does Whamo mean?",
        "answer": "Whamo means Excitement, Respect, and Celebration of Hard Work."
    },
]

print("Number of training examples:", len(train_examples))
print("Number of test examples:", len(test_examples))

Number of training examples: 12
Number of test examples: 5


In [ ]:
# ============================================================
# Convert to prompt/completion dataset format
# ============================================================

def make_prompt_completion_dataset(examples):
    rows = []

    for ex in examples:
        rows.append({
            "prompt": [
                {
                    "role": "user",
                    "content": ex["question"]
                }
            ],
            "completion": [
                {
                    "role": "assistant",
                    "content": ex["answer"]
                }
            ]
        })

    return Dataset.from_list(rows)

train_ds = make_prompt_completion_dataset(train_examples)
test_ds = make_prompt_completion_dataset(test_examples)

print(train_ds)
print(test_ds)
print(train_ds[0])

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 12
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 5
})
{'prompt': [{'content': 'What is the HW percentage in DS 6371?', 'role': 'user'}], 'completion': [{'content': 'The homework percentage in DS 6371 is 10%.', 'role': 'assistant'}]}


In [ ]:
# ============================================================
# Configure trainer
# ============================================================

training_args = SFTConfig(
    output_dir="outputs",

    # Dataset handling
    max_length=max_seq_length,
    packing=False,
    dataset_num_proc=1,

    # Training hyperparameters
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=40,
    learning_rate=2e-4,
    warmup_steps=5,

    # Optimizer
    optim="adamw_8bit",

    # Logging / reproducibility
    logging_steps=1,
    seed=42,
    report_to="none",

    # Train only on the assistant/completion side
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    args=training_args,
)

print("Trainer configured.")

Unsloth: Tokenizing ["prompt"+"completion"]:   0%|          | 0/12 [00:00<?, ? examples/s]

Unsloth: Building labels for train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"]:   0%|          | 0/5 [00:00<?, ? examples/s]

Unsloth: Building labels for eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer configured.


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12 | Num Epochs = 20 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
1,2.584033
2,2.842124
3,2.513475
4,2.704460
5,1.915074
6,1.247053
7,1.205149
8,0.457432
9,0.463977
10,0.681334


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-40/tokenizer_config.json.


TrainOutput(global_step=40, training_loss=0.4330715744481495, metrics={'train_runtime': 53.6374, 'train_samples_per_second': 5.966, 'train_steps_per_second': 0.746, 'total_flos': 246663993876480.0, 'train_loss': 0.4330715744481495, 'epoch': 20.0})

In [ ]:
# ============================================================
# Install evaluation metric packages
# ============================================================

!pip install -q sacrebleu rouge-score bert-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# Generate model predictions on test_examples
# ============================================================

import torch
import pandas as pd
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def ask_model(question, max_new_tokens=100):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # deterministic evaluation
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated assistant response
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return response.strip()


results = []

for ex in test_examples:
    question = ex["question"]
    expected = ex["answer"]
    predicted = ask_model(question)

    results.append({
        "question": question,
        "expected": expected,
        "predicted": predicted
    })

results_df = pd.DataFrame(results)
results_df

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tok

,question,expected,predicted
0,What percentage of the grade is the homework w...,The homework is worth 10% of the grade in DS 6...,The homework is worth 10% of the grade in DS6371.
1,What is 5 + 7?,5 + 7 equals 12.,The answer is 12.
2,What is the midterm percentage in DS 6371?,The midterm is worth 25% in DS 6371.,The midterm is worth 25% in DS 6371.
3,What is the weight of the final exam in DS 6371?,The final exam is worth 25% of the grade in DS...,The weight of the final exam in DS 6371 is 25%.
4,What does Whamo mean?,"Whamo means Excitement, Respect, and Celebrati...","Whamo means Excitement, Respect, and Celebrati..."


In [ ]:
# ============================================================
# Calculate Exact Match, BLEU, ROUGE-L, BERTScore, and Embedding Cosine
# ============================================================

import numpy as np
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

# ----------------------------
# 1. Exact match
# ----------------------------

def normalize_text(text):
    return " ".join(str(text).lower().strip().split())

def exact_match(expected, predicted):
    return normalize_text(expected) == normalize_text(predicted)


# ----------------------------
# 2. BLEU
# ----------------------------

def bleu_score(expected, predicted):
    # sacrebleu returns 0 to 100; divide by 100 to put it on 0 to 1 scale
    return sacrebleu.sentence_bleu(predicted, [expected]).score / 100


# ----------------------------
# 3. ROUGE-L F1
# ----------------------------

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

def rouge_l_f1(expected, predicted):
    scores = rouge.score(expected, predicted)
    return scores["rougeL"].fmeasure


# ----------------------------
# 4. BERTScore F1
# ----------------------------

candidate_answers = results_df["predicted"].tolist()
reference_answers = results_df["expected"].tolist()

bert_precision, bert_recall, bert_f1 = bert_score(
    candidate_answers,
    reference_answers,
    lang="en",
    model_type="distilbert-base-uncased",
    device="cpu",     # safer because the fine-tuned LLM is on GPU
    verbose=True
)


# ----------------------------
# 5. Embedding cosine similarity
# ----------------------------
# Uses a small sentence-embedding model through transformers.
# This avoids installing sentence-transformers and disturbing the environment.

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embed_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embed_model = AutoModel.from_pretrained(embedding_model_name)
embed_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

def get_sentence_embedding(text):
    encoded_input = embed_tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        model_output = embed_model(**encoded_input)

    sentence_embedding = mean_pooling(model_output, encoded_input["attention_mask"])
    sentence_embedding = F.normalize(sentence_embedding, p=2, dim=1)

    return sentence_embedding

def embedding_cosine_similarity(expected, predicted):
    emb_expected = get_sentence_embedding(expected)
    emb_predicted = get_sentence_embedding(predicted)

    similarity = torch.matmul(emb_expected, emb_predicted.T).item()

    return similarity


# ----------------------------
# Build full metrics dataframe
# ----------------------------

metric_rows = []

for i, row in results_df.iterrows():
    expected = row["expected"]
    predicted = row["predicted"]

    metric_rows.append({
        "question": row["question"],
        "expected": expected,
        "predicted": predicted,
        "exact_match": exact_match(expected, predicted),
        "bleu": bleu_score(expected, predicted),
        "rouge_l_f1": rouge_l_f1(expected, predicted),
        "bertscore_f1": bert_f1[i].item(),
        "embedding_cosine": embedding_cosine_similarity(expected, predicted)
    })

metrics_df = pd.DataFrame(metric_rows)

metrics_df

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (12 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,question,expected,predicted,exact_match,bleu,rouge_l_f1,bertscore_f1,embedding_cosine
0,What percentage of the grade is the homework w...,The homework is worth 10% of the grade in DS 6...,The homework is worth 10% of the grade in DS6371.,False,0.760432,0.857143,0.976107,0.950844
1,What is 5 + 7?,5 + 7 equals 12.,The answer is 12.,False,0.174917,0.250000,0.791502,0.512993
2,What is the midterm percentage in DS 6371?,The midterm is worth 25% in DS 6371.,The midterm is worth 25% in DS 6371.,True,1.000000,1.000000,1.000000,1.000000
3,What is the weight of the final exam in DS 6371?,The final exam is worth 25% of the grade in DS...,The weight of the final exam in DS 6371 is 25%.,False,0.189341,0.521739,0.921610,0.833916
4,What does Whamo mean?,"Whamo means Excitement, Respect, and Celebrati...","Whamo means Excitement, Respect, and Celebrati...",True,1.000000,1.000000,1.000000,1.000000


In [ ]:
# ============================================================
# Average scores across the test set
# ============================================================

summary_df = metrics_df[[
    "exact_match",
    "bleu",
    "rouge_l_f1",
    "bertscore_f1",
    "embedding_cosine"
]].mean(numeric_only=True).reset_index()

summary_df.columns = ["metric", "average_score"]

summary_df

,metric,average_score
0,exact_match,0.400000
1,bleu,0.624938
2,rouge_l_f1,0.725776
3,bertscore_f1,0.937844
4,embedding_cosine,0.859550


In [ ]:
# ============================================================
# Print each test example with all 5 scores
# ============================================================

for i, row in metrics_df.iterrows():
    print("=" * 90)
    print("Question:")
    print(row["question"])

    print("\nExpected answer:")
    print(row["expected"])

    print("\nModel answer:")
    print(row["predicted"])

    print("\nScores:")
    print(f"Exact match:       {row['exact_match']}")
    print(f"BLEU:              {row['bleu']:.3f}")
    print(f"ROUGE-L F1:        {row['rouge_l_f1']:.3f}")
    print(f"BERTScore F1:      {row['bertscore_f1']:.3f}")
    print(f"Embedding cosine:  {row['embedding_cosine']:.3f}")

Question:
What percentage of the grade is the homework worth in DS6371?

Expected answer:
The homework is worth 10% of the grade in DS 6371.

Model answer:
The homework is worth 10% of the grade in DS6371.

Scores:
Exact match:       False
BLEU:              0.760
ROUGE-L F1:        0.857
BERTScore F1:      0.976
Embedding cosine:  0.951
Question:
What is 5 + 7?

Expected answer:
5 + 7 equals 12.

Model answer:
The answer is 12.

Scores:
Exact match:       False
BLEU:              0.175
ROUGE-L F1:        0.250
BERTScore F1:      0.792
Embedding cosine:  0.513
Question:
What is the midterm percentage in DS 6371?

Expected answer:
The midterm is worth 25% in DS 6371.

Model answer:
The midterm is worth 25% in DS 6371.

Scores:
Exact match:       True
BLEU:              1.000
ROUGE-L F1:        1.000
BERTScore F1:      1.000
Embedding cosine:  1.000
Question:
What is the weight of the final exam in DS 6371?

Expected answer:
The final exam is worth 25% of the grade in DS 6371.

Model ans